In [1]:
!pip install polars

  Using cached polars-1.38.1-py3-none-any.whl.metadata (10 kB)
  Using cached polars_runtime_32-1.38.1-cp310-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (1.5 kB)
Using cached polars-1.38.1-py3-none-any.whl (810 kB)
Using cached polars_runtime_32-1.38.1-cp310-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (45.8 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [polars]2m1/2 [polars]


In [2]:
import polars as pl
import seaborn as sns
import matplotlib.pyplot as plt
import math
import pandas as pd

## Read data from S3 and join

In [ ]:
source_bucket = 's3://msds-26.2-data'
file_names = ['sanitized_2022.csv', 'sanitized_2023.csv', 'sanitized_2024.csv', 'sanitized_2025.csv']

In [ ]:
# Read csv from s3 and concat
df_recovery = pl.DataFrame()
for name in file_names:
    file_path = f'{source_bucket}/{name}'
    df_name = pl.read_csv(file_path, separator="\t")
    df_recovery = pl.concat([df_recovery, df_name])

In [ ]:
del df_name

In [ ]:
len(df_recovery)

## Data Cleaning

In [ ]:
# Drop columns where all values are null
cols_to_drop = (
    df_recovery.select(pl.all().is_null().all())
    .unpivot()
    .filter(pl.col("value") == True)
    .select("variable")
    .to_series()
    .to_list()
)

# Drop the identified columns from the original DataFrame
df_recovery = df_recovery.drop(cols_to_drop)

In [ ]:
# Remove all C-Returns from the data
df_recovery = df_recovery.filter(pl.col('recovery_type') != 'C-Returns')

In [ ]:
# Create target variable for propensity to waste model
df_recovery = df_recovery.with_columns(
    pl.when(pl.col('recovery_type') == 'Sales').then(pl.lit(0))
    .otherwise(pl.lit(1)).alias('recovery')
)

In [ ]:
df_recovery.filter(pl.col('country') != pl.col('marketplace_id'))

In [ ]:
# Remove marketplace_id b/c it's a duplicate of country
df_recovery = df_recovery.drop(['marketplace_id'])

In [ ]:
# Reorder the columns
df_recovery = df_recovery.select([
 'hashed_fc',
 'year',
 'month',
 'week',
 'gl_product_group',
 'product_type',
 'macro_category',
 'item_disposition_code',
 'reason_code',
 'application_name',
 'is_stranded',
 'reason_code_type',
 'reason_code_stranded',
 'stranded_potential_issue',
 'is_hazmat',
 'units',
 'cogs',
 'weight',
 'country',
 'country_state',
 'zip_code',
 'site_type',
 'site_category',
 'recovery',
 'recovery_type'])

In [ ]:
df_recovery.null_count()

In [ ]:
len(df_recovery)

In [ ]:
destination = "s3://msds-26.2-data/clean/combined_recovery_data.parquet"
df_recovery.write_parquet(destination)

In [4]:
# Remove 3000 isntances of missing gl_product_group
df_recovery = df_recovery.filter(pl.col('gl_product_group').is_not_null())

In [5]:
df_recovery.describe()

statistic,hashed_fc,year,month,week,gl_product_group,product_type,macro_category,item_disposition_code,reason_code,application_name,is_stranded,reason_code_type,reason_code_stranded,stranded_potential_issue,is_hazmat,units,cogs,weight,country,country_state,zip_code,site_type,site_category,recovery,recovery_type
str,str,f64,f64,f64,f64,str,str,str,str,str,str,str,str,str,str,f64,f64,f64,str,str,str,str,str,f64,str
"""count""","""55896510""",5.589651e7,5.589651e7,5.589651e7,5.589651e7,"""55896510""","""55896510""","""37622546""","""7686817""","""4180646""","""18273964""","""10889505""","""10889505""","""18273964""","""55896510""",5.589651e7,5.589651e7,5.5879163e7,"""55896510""","""51479053""","""51474827""","""51479053""","""51479053""",5.589651e7,"""55896510"""
"""null_count""","""0""",0.0,0.0,0.0,0.0,"""0""","""0""","""18273964""","""48209693""","""51715864""","""37622546""","""45007005""","""45007005""","""37622546""","""0""",0.0,0.0,17347.0,"""0""","""4417457""","""4421683""","""4417457""","""4417457""",0.0,"""0"""
"""mean""",null,2023.762283,6.523622,26.300421,219.716444,null,null,null,null,null,null,null,null,null,null,602.94825,4719.664789,653832.260101,null,null,null,null,null,0.599529,null
"""std""",null,1.0951,3.438518,14.998537,148.291072,null,null,null,null,null,null,null,null,null,null,5048.202846,58651.273689,4.5283e6,null,null,null,null,null,0.489994,null
"""min""","""0000c60d3b14250deb1312b21a448a…",2022.0,1.0,1.0,-1.0,"""Food""","""FBA""","""Overstock""","""A""","""DeleteItemsApp""","""N""","""ALSI""","""ABP_LOGO_MISUSE""","""N""","""N""",4.04,1.66,2.53,"""CA""","""AB""","""01581""","""3rd Party Logistics (3PL)""","""AMXL""",0.0,"""Bintool Donations"""
"""25%""",null,2023.0,4.0,13.0,86.0,null,null,null,null,null,null,null,null,null,null,5.32,1.68,1108.77,null,null,null,null,null,0.0,null
"""50%""",null,2024.0,7.0,26.0,199.0,null,null,null,null,null,null,null,null,null,null,9.08,1.69,10838.43,null,null,null,null,null,1.0,null
"""75%""",null,2025.0,10.0,39.0,309.0,null,null,null,null,null,null,null,null,null,null,54.3,122.85,72183.5,null,null,null,null,null,1.0,null
"""max""","""fffda6de5ef71646faf113a5359c32…",2025.0,12.0,52.0,814.0,"""Pet Food""","""RETAIL""","""Unsellable""","""W""","""WAVE_DPT""","""Y""","""\N""","""\N""","""Y""","""Y""",850408.98,3.5082e7,3.7172e8,"""US""","""WV""","""V6W 0C6""","""Zappos""","""Other""",1.0,"""Warehouse Deals and G&R"""


In [14]:
df_recovery.filter(pl.col('recovery') == 0)['reason_code'].value_counts()
# Only products that enter the recovery funnel have reason code - might not be usable for propensity to waste model

reason_code,count
str,u32
null,22384933


In [15]:
df_recovery.filter(pl.col('recovery') == 0)['is_stranded'].value_counts()
# Only products that enter the recovery funnel are stranded

is_stranded,count
str,u32
null,22384933


In [16]:
df_recovery_agg = df_recovery.group_by(['hashed_fc', 'year', 'month', 'week', 'gl_product_group']).agg(
    pl.len().alias('num_records'),
    ((pl.col('reason_code') == 'E').sum()/pl.len()).alias('proportion_reason_code_E'),
    ((pl.col('reason_code') == 'O').sum()/pl.len()).alias('proportion_reason_code_O'),
    ((pl.col('macro_category') == 'RETAIL').sum()/pl.len()).alias('proportion_macro_category_RETAIL'),
    ((pl.col('macro_category') == 'FBA').sum()/pl.len()).alias('proportion_macro_category_FBA'),
    ((pl.col('is_hazmat') == 'O').sum()/pl.len()).alias('proportion_hazmat'),
    ((pl.col('product_type') == 'Food').sum()/pl.len()).alias('proportion_food'),
    ((pl.col('product_type') == 'Non Food').sum()/pl.len()).alias('proportion_non_food'),
    ((pl.col('product_type') == 'Pet Food').sum()/pl.len()).alias('proportion_pet_food'),
    ((pl.col('is_stranded') == 'Y').sum()/pl.len()).alias('proportion_stranded'),
    ((pl.col('reason_code_type') == 'LLSI').sum()/pl.len()).alias('proportion_reason_code_type_LLSI'),
    ((pl.col('reason_code_type') == '\\N').sum()/pl.len()).alias('proportion_reason_code_type_N'),
    ((pl.col('reason_code_type') == 'MCF').sum()/pl.len()).alias('proportion_reason_code_type_MCF'),
    ((pl.col('reason_code_type') == 'ALSI').sum()/pl.len()).alias('proportion_reason_code_type_ALSI'),
    pl.col('units').sum().alias('units_total'),
    pl.col('units').mean().alias('units_mean'),
    pl.col('units').std().alias('units_std'),
    pl.col('cogs').sum().alias('cogs_total'),
    pl.col('cogs').mean().alias('cogs_mean'),
    pl.col('cogs').std().alias('cogs_std'),
    pl.col('weight').sum().alias('weight_total'),
    pl.col('weight').mean().alias('weight_mean'),
    pl.col('weight').std().alias('weight_std'),
    pl.min('country'),
    pl.min('country_state'),
    pl.min('zip_code'),
    pl.min('site_type'),
    pl.min('site_category'),
    (pl.sum('recovery')/pl.len()).alias('proportion_recovery')
).sort(by='num_records', descending=True)
df_recovery_agg

hashed_fc,year,month,week,gl_product_group,num_records,proportion_reason_code_E,proportion_reason_code_O,proportion_macro_category_RETAIL,proportion_macro_category_FBA,proportion_hazmat,proportion_food,proportion_non_food,proportion_pet_food,proportion_stranded,proportion_reason_code_type_LLSI,proportion_reason_code_type_N,proportion_reason_code_type_MCF,proportion_reason_code_type_ALSI,units_total,units_mean,units_std,cogs_total,cogs_mean,cogs_std,weight_total,weight_mean,weight_std,country,country_state,zip_code,site_type,site_category,proportion_recovery
str,i64,i64,f64,f64,u32,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,str,str,str,str,f64
"""825f65d7988f020fca7a29b867278d…",2024,6,23.0,194.0,153,0.045752,0.03268,0.267974,0.732026,0.0,0.0,1.0,0.0,0.614379,0.51634,0.0,0.026144,0.071895,210272.03,1374.326993,11608.679603,520937.58,3404.820784,35465.265099,7.2228e7,472080.414314,3.7772e6,"""CA""","""CA""","""91752""","""AR Sortable""","""GCF""",0.941176
"""05a669864a534498bb4f40f3b742c4…",2024,3,13.0,194.0,150,0.006667,0.006667,0.233333,0.766667,0.0,0.0,1.0,0.0,0.66,0.52,0.006667,0.053333,0.08,29276.14,195.174267,1121.530992,116532.64,776.884267,7634.194029,1.0380e7,69202.653933,380112.391693,"""US""","""AZ""","""85043""","""AR Sortable""","""GCF""",0.973333
"""05a669864a534498bb4f40f3b742c4…",2024,3,10.0,201.0,149,0.0,0.0,0.134228,0.865772,0.0,0.0,1.0,0.0,0.778523,0.624161,0.0,0.053691,0.100671,43539.76,292.213154,1735.165378,10654.46,71.506443,366.875385,6.6145e7,443923.814832,2.6670e6,"""US""","""AZ""","""85043""","""AR Sortable""","""GCF""",0.979866
"""05a669864a534498bb4f40f3b742c4…",2024,3,11.0,201.0,148,0.0,0.0,0.162162,0.837838,0.0,0.0,1.0,0.0,0.75,0.614865,0.013514,0.054054,0.067568,40617.88,274.445135,1690.933766,17117.46,115.658514,642.113654,6.1798e7,417557.330473,2.6615e6,"""US""","""AZ""","""85043""","""AR Sortable""","""GCF""",0.97973
"""825f65d7988f020fca7a29b867278d…",2024,2,8.0,121.0,145,0.034483,0.02069,0.268966,0.731034,0.0,0.0,1.0,0.0,0.586207,0.434483,0.0,0.013793,0.137931,191943.29,1323.746828,10256.68342,943451.91,6506.564897,73662.832535,1.6130e8,1.1124e6,9.6061e6,"""CA""","""CA""","""91752""","""AR Sortable""","""GCF""",0.931034
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""5582e5c4aaa5d1a475e07fe002aa46…",2022,9,36.0,21.0,1,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,7.77,7.77,null,1.68,1.68,null,2301.37,2301.37,null,"""MX""",null,null,null,null,0.0
"""f5d61f51d1645bbd602cdf1595b964…",2023,9,36.0,121.0,1,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,5.31,5.31,null,12.19,12.19,null,2058.92,2058.92,null,"""US""",null,null,null,null,0.0
"""70332d66a560ee6818f1dde2a52871…",2022,12,49.0,194.0,1,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,4.05,4.05,null,3.65,3.65,null,109.39,109.39,null,"""US""",null,null,null,null,0.0


In [17]:
destination = "s3://msds-26.2-data/clean/combined_recovery_data_aggregated.parquet"
df_recovery_agg.write_parquet(destination)